# Chapter 08: Variables, Secrets, Configuration and Environments (notebook edition)

## Learning Objectives

- Resolve env scope
- Explain what masking does not cover
- Read environment overrides and approval waits

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

GHA_MODE = os.environ.get("GHA_MODE", "fixture")  # "fixture" | "live"
GHA_REPO = os.environ.get("GHA_REPO", "Friend09/practice_git_intro_to_github_actions")
print("mode:", GHA_MODE, "| repo:", GHA_REPO)

## 1. Scope resolution

The closest `env` definition wins. We model it and compare with what the real run printed.

In [ ]:
import json
def resolve(name, workflow, job, step):
    for scope in (step, job, workflow):
        if name in scope:
            return scope[name]
cfg = json.loads((ROOT/"fixtures"/"config_ch08_run.json").read_text())["reports"]
wf, job = {"SCOPE": "workflow"}, {"SCOPE": "job"}
print(resolve("SCOPE", wf, job, {"SCOPE": "step"}), resolve("SCOPE", wf, job, {}), resolve("SCOPE", wf, {}, {}))
assert cfg["scopes"]["scope_at_step"] == "step" and cfg["scopes"]["scope_at_job"] == "job"
assert cfg["workflow_scope_only"]["scope_at_workflow"] == "workflow"

## 2. GITHUB_ENV timing

A value written to `GITHUB_ENV` is invisible to the writing step and visible to the next.

In [ ]:
s = cfg["scopes"]
print(s["same_step_sees"], "|", s["next_step_sees"])
assert s["same_step_sees"] == "unset" and s["next_step_sees"] == "written-by-previous-step"

## 3. What masking covered

The dummy secret printed as `***` directly and in base64, but its reversal leaked until `::add-mask::`.

In [ ]:
v = cfg["vars_and_secrets"]
for k in ["secret_direct", "secret_base64", "derived_before_mask", "derived_after_mask"]:
    print(f"{k:22}", v[k])
assert v["derived_before_mask"] == "tally-demo-s3cret-VALUE"[::-1]
assert v["derived_after_mask"] == "***"

## 4. Variable and secret scope

The staging environment's variable overrides the repo's; its secret is invisible elsewhere.

In [ ]:
envr = json.loads((ROOT/"fixtures"/"environments_ch08_run.json").read_text())["reports"]
print(v["repo_color"], envr["staging"]["staging_color"], v["env_secret_not_visible_outside_env_len"], envr["staging"]["staging_only_len"])
assert (v["repo_color"], envr["staging"]["staging_color"]) == ("blue", "green")
assert v["env_secret_not_visible_outside_env_len"] == "0"

## 5. The approval wait

Staging finished 13:28:46; production started 13:29:37.

In [ ]:
from datetime import datetime
f = "%H:%M:%S"
wait = (datetime.strptime("13:29:37", f) - datetime.strptime("13:28:46", f)).seconds
print(wait, "seconds waiting (unbilled)")
assert wait == 51

## Takeaways & Next Steps

- Closest `env` wins; `GITHUB_ENV` applies to later steps.
- Mask derived secret values explicitly.
- Environments gate secrets and approvals.
- Next: Chapter 09, moving data between steps and jobs.

## Chapter Link

Read: `learning_modules/chapter_08_variables_secrets_environments.md`